# Setup
Run using the scenic environment specified by pixi

In [7]:
# base
import os
import subprocess
import gc
from pathlib import Path

# data manipulation
import numpy as np
import pandas as pd

# single cell
import anndata as ad
import scanpy as sc
import loompy as lp

gc.collect()

20022

### Get Databases
https://resources.aertslab.org/cistarget/

In [ ]:
# ! wget -nc https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl
# ! wget -nc https://resources.aertslab.org/cistarget/tf_lists/allTFs_mm.txt
# ! wget -nc https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather
# ! wget -nc https://resources.aertslab.org/cistarget/databases/mus_musculus/mm10/refseq_r80/mc_v10_clust/gene_based/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather

# Scenic Prep

In [ ]:
# path to unfiltered loom file (this will be created in the optional steps below)
CORES = 10
DATADIR = Path("../../../data")
OUTSDIR = Path("../../../outs")
REFDIR = Path("../../../references")
MAIN_DIR = DATADIR / "processed" / "single_cell" / "combined"
TOOL_DIR = MAIN_DIR / "tools" / "scenic"

METADATA = ["Diet", "Age", "Depot", "Sex"]
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]
INT_KEY = "INT_scvi_hvg-Identifier"

GRN_RANKINGS = " ".join(map(str, (REFDIR / "scenic").glob("*feather")))
MOTIF_ANNOTS = str(REFDIR / "scenic" / "motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl")
TF_LIST = str(REFDIR / "scenic" / "mm_mgi_tfs.txt")

path_loom_input = str(TOOL_DIR / "pyscenic_input.loom")
path_adjacency = str(TOOL_DIR / "scenic_adj.csv")
path_regulon_pred = str(TOOL_DIR / "scenic_reg.csv")
path_aucell_out = str(TOOL_DIR / "aucell_mtx.csv")
path_loom_output = str(TOOL_DIR / "pyscenic_output.loom")

# Set maximum number of jobs for Scanpy.
sc.settings.njobs = CORES

In [75]:
annotation = "eWAT_Male_SLIM.h5ad"
adata = ad.read_h5ad(DATADIR / "processed" / "single_cell" / "combined" / annotation)

# percentiles = adata.obs["n_genes"].quantile([0.01, 0.05, 0.10, 0.50, 1])
# print(percentiles)

# # create loompy
# lp.create(path_loom_input, adata.X.transpose(), {"Gene": np.array(adata.var.index),}, {"CellID": np.array(adata.obs.index)})

In [ ]:
# run grnboost
outfile = OUTSDIR / "scenic-1_grnboost_adjacency.out"
if not Path(path_adjacency).exists():
    subprocess.run(
        [
            "pyscenic",
            "grn",
            path_loom_input,
            TF_LIST,
            "--num_workers",
            CORES,
            "--output",
            path_adjacency,
            "&>",
            outfile,
        ],
        check=True,
    )

# pyscenic grn "data/processed/single_cell/combined/tools/scenic/input.loom" "references/scenic/mm_mgi_tfs.txt" --output "data/processed/single_cell/combined/tools/scenic/scenic_adj.csv" --num_workers 20 &>> "outs/single_cell/tools/pyscenic.out"

results_adjacencies = pd.read_csv(path_adjacency, index_col=False, sep=",")
print(f"Number of associations: {results_adjacencies.shape[0]}")
print(results_adjacencies.head())

In [ ]:
# run ctx regulon predicton

outfile = OUTSDIR / "scenic-2_ctx.out"
if not Path(path_regulon_pred).exists():
    subprocess.run(
        [
            "time" "pyscenic",
            "ctx",
            path_adjacency,
            str(GRN_RANKINGS),
            "--annotations_fname",
            MOTIF_ANNOTS,
            "--expression_mtx_fname",
            path_loom_input,
            "--output",
            path_regulon_pred,
            "--num_workers",
            CORES,
            "--mask_dropouts",
            "&>",
            outfile,
        ],
        check=True,
    )

# pyscenic ctx "data/processed/single_cell/combined/tools/scenic/scenic_adj.csv" \
#     "references/scenic/mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather" \
#     "references/scenic/mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather" \
#     --annotations_fname references/scenic/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl \
#     --expression_mtx_fname data/processed/single_cell/combined/tools/scenic/pyscenic_input.loom \
#     --output data/processed/single_cell/combined/tools/scenic/scenic_reg.csv \
#     --mask_dropouts \
#     --num_workers 20 &>> "outs/single_cell/tools/pyscenic.out"

In [ ]:
# run aucell summary

outfile = OUTSDIR / "scenic-3_aucell.out"
if not Path(path_aucell_out).exists():
    subprocess.run(
        [
            "pyscenic",
            "aucell",
            path_loom_input,
            path_regulon_pred,
            "--output",
            path_aucell_out,
            "--num_workers",
            CORES,
            "&>",
            outfile,
        ],
        check=True,
    )

# pyscenic aucell \
#     "data/processed/single_cell/combined/tools/scenic/pyscenic_input.loom" \
#     "data/processed/single_cell/combined/tools/scenic/scenic_reg.csv" \
#     --output "data/processed/single_cell/combined/tools/scenic/aucell_mtx.csv" \
#     --num_workers 20 &>> "outs/single_cell/tools/pyscenic.out"

Then, run TSNE/UMAP on SCENIC AUCell space. See the scverse best practices link [here](https://www.sc-best-practices.org/mechanisms/gene_regulatory_networks.html#takeaways) for more details.